# 🔬 Lab W6-3 — เมื่อใดควรทิ้งผลการแบ่งกลุ่ม

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 6 — Data Mining II**

Lab นี้ใช้คู่กับสื่อจำลอง **Cluster Reality Check** (`/sims/cluster-reality-check`)
ค่า silhouette และการเลือก k ในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง
(ดูหมายเหตุเรื่อง inertia ท้ายเล่ม)

## สิ่งที่จะได้เรียนรู้
1. พิสูจน์ว่า **K-Means คืนกลุ่มมาให้เสมอ** แม้ข้อมูลไม่มีโครงสร้างเลย
2. ใช้ **ข้อมูลอ้างอิงแบบสุ่ม** เป็นเกณฑ์เทียบก่อนเชื่อผลการแบ่งกลุ่ม
3. วัด **ความเสถียรของกลุ่ม** ด้วย Adjusted Rand Index
4. เขียน **เกณฑ์การยอมรับ** ที่ทีมใช้ได้จริงก่อนนำผลไปเสนอผู้บริหาร

## ข้อมูล
* `customers_rfm.csv` — ลูกค้าจริง 1,200 ราย ที่มีกลุ่มพฤติกรรมอยู่จริง
* `no_structure.csv` — จุดสุ่มสม่ำเสมอ 900 จุด **ที่ไม่มีกลุ่มอยู่จริงเลย**

ทั้งสองไฟล์มีชื่อคอลัมน์เหมือนกันทุกประการ จึงสลับใช้ได้โดยไม่ต้องแก้โค้ดแม้แต่บรรทัดเดียว

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

BASE = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
        "master/datasets/week06/")
F = ["recency_days", "frequency", "monetary"]

real = pd.read_csv(BASE + "customers_rfm.csv").sort_values("customer_id").reset_index(drop=True)
noise = pd.read_csv(BASE + "no_structure.csv").sort_values("customer_id").reset_index(drop=True)

X_real = StandardScaler().fit_transform(real[F].astype(float))
X_noise = StandardScaler().fit_transform(noise[F].astype(float))

print(f"ข้อมูลลูกค้าจริง : {X_real.shape}")
print(f"จุดสุ่มล้วน      : {X_noise.shape}")
print("\nทั้งสองชุดผ่านการปรับสเกลแบบเดียวกัน และจะถูกวิเคราะห์ด้วยโค้ดบรรทัดเดียวกัน")

## ส่วนที่ 1 — รันแบบเดียวกันกับทั้งสองชุด

### 🧑‍💻 งานที่ 1
เขียนฟังก์ชัน `sweep(X)` ที่คืน `DataFrame` ของ inertia และ silhouette
สำหรับ k = 2 ถึง 8 แล้วรันกับทั้งสองชุดข้อมูล

**อย่าเพิ่งดูว่าชุดไหนเป็นชุดไหน** — ลองอ่านตัวเลขก่อนแล้วเดาว่าชุดใดมีโครงสร้างจริง

*เฉลยที่ถูกต้อง: ข้อมูลจริง silhouette สูงสุด 0.5847 ที่ k = 4
ส่วนจุดสุ่มได้ 0.2868 ที่ k = 6*

In [ ]:
def sweep(X, label):
    rows = []
    for k in range(2, 9):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        sizes = np.bincount(km.labels_, minlength=k)
        rows.append({
            "k": k, "inertia": km.inertia_,
            "silhouette": silhouette_score(X, km.labels_),
            "กลุ่มเล็กสุด": sizes.min(), "กลุ่มใหญ่สุด": sizes.max(),
            "อัตราส่วนขนาด": sizes.max() / sizes.min(),
        })
    out = pd.DataFrame(rows).set_index("k")
    print(f"\n--- {label} ---")
    print(out.to_string())
    print(f"silhouette สูงสุด {out.silhouette.max():.4f} ที่ k = {out.silhouette.idxmax()}")
    print(f"ช่วงห่างของ silhouette ตลอด k = {out.silhouette.max()-out.silhouette.min():.4f}")
    return out


s_real = sweep(X_real, "ชุด A")
s_noise = sweep(X_noise, "ชุด B")

## ส่วนที่ 2 — สัญญาณสี่ข้อที่แยกทั้งสองชุดออกจากกัน

### 🧑‍💻 งานที่ 2
สร้างตารางเปรียบเทียบสัญญาณต่อไปนี้ระหว่างสองชุด

1. ค่า silhouette สูงสุด
2. ช่วงห่างระหว่าง silhouette สูงสุดกับต่ำสุด
3. อัตราส่วนระหว่างกลุ่มใหญ่สุดกับกลุ่มเล็กสุด ที่ k ที่ดีที่สุด
4. เส้น silhouette มี "ยอด" ที่ชัดเจนหรือไม่

แล้วอธิบายว่าเหตุใดสัญญาณข้อ 2 และ 3 จึงบอกได้มากกว่าข้อ 1

In [ ]:
def signals(s, label):
    k = s.silhouette.idxmax()
    return {
        "ชุด": label,
        "silhouette สูงสุด": s.silhouette.max(),
        "ที่ k": k,
        "ช่วงห่าง": s.silhouette.max() - s.silhouette.min(),
        "อัตราส่วนขนาดกลุ่ม": s.loc[k, "อัตราส่วนขนาด"],
    }


cmp = pd.DataFrame([signals(s_real, "A (ลูกค้าจริง)"),
                    signals(s_noise, "B (สุ่มล้วน)")]).set_index("ชุด")
print(cmp.to_string())

print(f"""
เหตุใดข้อ 2 และ 3 จึงบอกได้มากกว่าข้อ 1
------------------------------------------
ข้อ 1 (ค่าสูงสุด) : ชุด B ได้ {s_noise.silhouette.max():.4f} ซึ่งตำราหลายเล่มจัดว่า 'พอใช้ได้'
                   ถ้าดูแค่ตัวเลขนี้ นักศึกษาจำนวนมากจะยอมรับแล้วเขียนรายงานต่อ

ข้อ 2 (ช่วงห่าง)  : ชุด A ต่าง {cmp.loc['A (ลูกค้าจริง)','ช่วงห่าง']:.4f}
                   ชุด B ต่างเพียง {cmp.loc['B (สุ่มล้วน)','ช่วงห่าง']:.4f}
                   ข้อมูลที่มีโครงสร้างจริงจะ 'ตอบสนอง' ต่อการเปลี่ยน k อย่างชัดเจน
                   ส่วนข้อมูลสุ่มให้ค่าใกล้เคียงกันไม่ว่าจะหั่นกี่ชิ้น
                   เพราะการหั่นพื้นที่ว่างเป็น k ส่วนก็ได้ผลพอกันทุกแบบ

ข้อ 3 (อัตราส่วนขนาด) : ชุด A ได้ {cmp.loc['A (ลูกค้าจริง)','อัตราส่วนขนาดกลุ่ม']:.2f}
                        ชุด B ได้ {cmp.loc['B (สุ่มล้วน)','อัตราส่วนขนาดกลุ่ม']:.2f}
                        กลุ่มจริงมีขนาดไม่เท่ากันเพราะสะท้อนสัดส่วนของประชากรจริง
                        ส่วนการหั่นพื้นที่ว่างจะได้ชิ้นขนาดใกล้เคียงกันเสมอ
                        กลุ่มที่ 'ขนาดเท่ากันสวยงามเกินไป' เป็นสัญญาณอันตราย
""")

## ส่วนที่ 3 — ความเสถียรของกลุ่ม

ถ้ากลุ่มมีอยู่จริง การสุ่มตัวอย่างมาแค่บางส่วนก็ยังควรพบกลุ่มเดิม

### 🧑‍💻 งานที่ 3
สุ่มข้อมูลย่อย 80% จำนวน 10 รอบ (ใช้ `random_state` ต่างกันทุกรอบ)
รัน K-Means ที่ k ที่ดีที่สุดของแต่ละชุด แล้ววัดว่าการจัดกลุ่มของแต่ละรอบ
ตรงกับการจัดกลุ่มของโมเดลที่ฝึกด้วยข้อมูลเต็มมากแค่ไหน ด้วย **Adjusted Rand Index**

ARI = 1 คือเหมือนกันทุกประการ · ARI ≈ 0 คือตรงกันแค่ระดับบังเอิญ

In [ ]:
def stability(X, k, rounds=10):
    full = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    scores = []
    for seed in range(rounds):
        rs = np.random.RandomState(seed)
        idx = rs.choice(len(X), size=int(len(X) * 0.8), replace=False)
        sub = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X[idx])
        scores.append(adjusted_rand_score(full.labels_[idx], sub.labels_))
    return np.array(scores)


k_real = int(s_real.silhouette.idxmax())
k_noise = int(s_noise.silhouette.idxmax())

for X, k, label in [(X_real, k_real, "A (ลูกค้าจริง)"), (X_noise, k_noise, "B (สุ่มล้วน)")]:
    sc = stability(X, k)
    print(f"{label}  k={k}")
    print(f"  ARI เฉลี่ย = {sc.mean():.4f} · ต่ำสุด = {sc.min():.4f} · "
          f"ส่วนเบี่ยงเบน = {sc.std():.4f}")

print("""
การตีความ
---------
ชุด A ให้ ARI สูงและเบี่ยงเบนน้อย — ตัดข้อมูลออก 20% แล้วยังได้กลุ่มเดิม
        แปลว่ากลุ่มเป็นคุณสมบัติของ 'ประชากร' ไม่ใช่ของ 'ตัวอย่างชุดนี้'

ชุด B ให้ ARI ต่ำและแกว่ง — เปลี่ยนตัวอย่างนิดเดียวกลุ่มก็เปลี่ยนไปคนละแบบ
        แปลว่าเส้นแบ่งที่ได้เป็นเพียงผลของเสียงรบกวน ไม่ใช่โครงสร้างจริง

ความเสถียรเป็นสัญญาณที่ 'ปลอมยากที่สุด' ในบรรดาสัญญาณทั้งสี่
เพราะข้อมูลที่ไม่มีโครงสร้างไม่มีทางให้ผลเสถียรได้เลย
""")

## ส่วนที่ 4 — K-Means บอกว่า "ไม่มีกลุ่ม" ไม่ได้

### 🧑‍💻 งานที่ 4
ลองรัน DBSCAN กับทั้งสองชุด แล้วเทียบว่ามันตอบต่างจาก K-Means อย่างไร

แล้วตอบว่าเหตุใด DBSCAN จึงบอกได้ว่า "ไม่มีกลุ่ม" ในขณะที่ K-Means บอกไม่ได้

In [ ]:
from sklearn.cluster import DBSCAN

for X, label in [(X_real, "A (ลูกค้าจริง)"), (X_noise, "B (สุ่มล้วน)")]:
    db = DBSCAN(eps=0.55, min_samples=12).fit(X)
    n_cl = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise = int((db.labels_ == -1).sum())
    print(f"{label:<18} DBSCAN พบ {n_cl} กลุ่ม · "
          f"จุดที่ถือเป็น noise {n_noise:,} จาก {len(X):,} "
          f"({n_noise/len(X)*100:.1f}%)")

print("""
เหตุใด DBSCAN จึงตอบได้
------------------------
K-Means มีข้อบังคับว่า 'ทุกจุดต้องอยู่ในกลุ่มใดกลุ่มหนึ่ง' และ 'ต้องมี k กลุ่มพอดี'
เมื่อสั่ง k = 5 มันจึงคืน 5 กลุ่มเสมอ ไม่ว่าจะป้อนอะไรเข้าไป
มันไม่มีคำว่า 'ไม่มี' อยู่ในคำตอบที่เป็นไปได้เลย

DBSCAN นิยามกลุ่มจาก 'ความหนาแน่น' — บริเวณที่มีจุดหนาแน่นเกินเกณฑ์จึงนับเป็นกลุ่ม
ถ้าไม่มีบริเวณใดหนาแน่นพอ มันจะตอบว่าไม่มีกลุ่มและจัดทุกจุดเป็น noise ได้
ความสามารถในการตอบว่า 'ไม่' คือสิ่งที่ทำให้มันตรวจสอบได้

ข้อแลกเปลี่ยน: DBSCAN ต้องจูน eps และ min_samples ซึ่งไวต่อสเกลไม่แพ้ K-Means
และทำงานได้แย่ลงมากเมื่อมิติสูงขึ้น จึงไม่ใช่ทางออกสำเร็จรูป
""")

## ส่วนที่ 5 — เกณฑ์การยอมรับ

### 🧑‍💻 งานที่ 5
เขียนฟังก์ชัน `should_i_trust(X, k)` ที่ตรวจสัญญาณทั้งหมดที่เรียนมา
แล้วคืนคำตอบว่า "ยอมรับ" หรือ "ทิ้ง" พร้อมเหตุผลรายข้อ

ทดสอบกับทั้งสองชุด แล้วยืนยันว่าฟังก์ชันตอบถูกทั้งคู่

In [ ]:
def should_i_trust(X, k, name=""):
    """เกณฑ์การยอมรับผลการแบ่งกลุ่ม — ต้องผ่านอย่างน้อย 3 ใน 4 ข้อ"""
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)

    # เกณฑ์เทียบ: ข้อมูลสุ่มที่มีช่วงเท่ากันทุกแกน
    rs = np.random.RandomState(0)
    ref = rs.uniform(X.min(0), X.max(0), size=X.shape)
    ref_sil = silhouette_score(ref, KMeans(n_clusters=k, random_state=42,
                                           n_init=10).fit_predict(ref))

    curve = [silhouette_score(X, KMeans(n_clusters=kk, random_state=42,
                                        n_init=10).fit_predict(X))
             for kk in range(2, 9)]
    sizes = np.bincount(km.labels_, minlength=k)
    ari = stability(X, k, rounds=5).mean()

    checks = [
        ("silhouette สูงกว่าข้อมูลสุ่มอ้างอิงอย่างน้อย 0.15",
         sil - ref_sil >= 0.15, f"{sil:.4f} เทียบ {ref_sil:.4f} (ต่าง {sil-ref_sil:+.4f})"),
        ("เส้น silhouette มียอดชัดเจน (ช่วงห่าง ≥ 0.08)",
         max(curve) - min(curve) >= 0.08, f"ช่วงห่าง {max(curve)-min(curve):.4f}"),
        ("ขนาดกลุ่มไม่เท่ากันเกินไป (อัตราส่วน ≥ 1.5)",
         sizes.max() / sizes.min() >= 1.5, f"อัตราส่วน {sizes.max()/sizes.min():.2f}"),
        ("กลุ่มเสถียรเมื่อสุ่มข้อมูลย่อย (ARI ≥ 0.6)",
         ari >= 0.6, f"ARI เฉลี่ย {ari:.4f}"),
    ]

    passed = sum(ok for _, ok, _ in checks)
    print(f"\n{'='*62}\nตรวจผลการแบ่งกลุ่ม: {name} (k={k})\n{'='*62}")
    for text, ok, detail in checks:
        print(f"  [{'✓' if ok else '✗'}] {text}\n      {detail}")
    verdict = "ยอมรับได้" if passed >= 3 else "ควรทิ้งผลนี้"
    print(f"\n  ผ่าน {passed}/4 ข้อ → {verdict}")
    return bool(passed >= 3)


assert should_i_trust(X_real, k_real, "ชุด A — ลูกค้าจริง")
assert not should_i_trust(X_noise, k_noise, "ชุด B — สุ่มล้วน")
print("\n✓ ฟังก์ชันตัดสินถูกทั้งสองชุด")

## ส่วนที่ 6 — เขียนเป็นนโยบายของทีม

### 🧑‍💻 งานที่ 6 (เขียนเป็นข้อความ)

1. เขียนเกณฑ์การยอมรับผลการแบ่งกลุ่มเป็นรายการที่ทีมคุณจะใช้กับทุกโปรเจกต์
   โดยต้องมีทั้งเกณฑ์เชิงสถิติและเกณฑ์เชิงธุรกิจ
2. ถ้าผลไม่ผ่านเกณฑ์ แต่ผู้บริหารกำลังรอผลอยู่ คุณจะสื่อสารอย่างไร
   เขียนเป็นข้อความจริงที่จะส่งไป ไม่เกิน 5 บรรทัด
3. ยกตัวอย่างสถานการณ์ที่ "ไม่มีกลุ่ม" เป็นคำตอบที่ **มีประโยชน์** ต่อธุรกิจ

In [ ]:
print("""ตัวอย่างคำตอบข้อ 2 — ข้อความถึงผู้บริหาร
=========================================
"ผลวิเคราะห์เสร็จแล้วครับ แต่มีข้อค้นพบที่ต้องรายงานก่อน

เราทดสอบด้วยการรันวิธีเดียวกันกับข้อมูลสุ่มที่ไม่มีโครงสร้างเลย
แล้วได้คะแนนใกล้เคียงกับข้อมูลจริงของเรา แปลว่ากลุ่มที่ได้ไม่น่าเชื่อถือ

ข้อเสนอ: ลูกค้าชุดนี้อาจมีพฤติกรรมเป็นสเปกตรัมต่อเนื่อง ไม่ได้แยกเป็นกลุ่ม
ถ้าเป็นเช่นนั้น การแบ่งช่วงตามตัวแปรเดียวที่ธุรกิจเข้าใจอยู่แล้วจะใช้งานได้จริงกว่า
ขอเวลา 3 วันทดสอบแนวทางนี้ครับ"

หลักการ: รายงานสิ่งที่พบพร้อมทางออกเสมอ ไม่ใช่รายงานแค่ว่าทำไม่ได้
และไม่ส่งผลที่รู้ว่าไม่น่าเชื่อถือไปให้คนอื่นตัดสินใจ

ตัวอย่างคำตอบข้อ 3 — เมื่อ 'ไม่มีกลุ่ม' มีประโยชน์
====================================================
1. ลูกค้าเป็นสเปกตรัมต่อเนื่อง ไม่ใช่กลุ่มที่แยกจากกัน
   → เลิกทำ segmented marketing แล้วเปลี่ยนไปทำ personalization รายบุคคล
     ประหยัดงบการสร้างแคมเปญ 5 แบบที่ไม่มีใครแตกต่างกันจริง

2. สาขาทุกแห่งมีพฤติกรรมเหมือนกัน
   → ใช้นโยบายเดียวทั้งเครือได้ ไม่ต้องลงทุนระบบบริหารแบบแยกกลุ่ม
     เป็นการประหยัดที่วัดเป็นเงินได้ทันที

3. ธุรกรรมที่สงสัยว่าทุจริตไม่รวมกลุ่มกัน
   → แปลว่าไม่ใช่ขบวนการที่มีรูปแบบ แต่เป็นการกระทำรายบุคคลแบบกระจาย
     กลยุทธ์การรับมือต้องเปลี่ยนจากการปิดกลุ่ม เป็นการคัดกรองรายรายการ

'ไม่มีกลุ่ม' จึงเป็นข้อค้นพบเชิงบวก ไม่ใช่ความล้มเหลวของงานวิเคราะห์
สิ่งที่ล้มเหลวคือการรายงานกลุ่มที่ไม่มีอยู่จริงว่ามีอยู่
""")

---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — sweep ทั้งสองชุดและได้ตัวเลขตรงเฉลย | 3 |
| งานที่ 2 — เปรียบเทียบสัญญาณ 4 ข้อพร้อมคำอธิบาย | 4 |
| งานที่ 3 — วัดความเสถียรด้วย ARI | 3 |
| งานที่ 4 — เทียบกับ DBSCAN และอธิบายความต่างเชิงหลักการ | 3 |
| งานที่ 5 — ฟังก์ชันเกณฑ์การยอมรับที่ตัดสินถูกทั้งสองชุด | 4 |
| งานที่ 6 — นโยบายของทีมและการสื่อสารกับผู้บริหาร | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/cluster-reality-check`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง

> 💡 **หมายเหตุเรื่องการเทียบตัวเลขกับสื่อจำลอง**
> สื่อจำลองใช้ K-Means ที่กำหนดค่าเริ่มต้นของ centroid แบบตายตัว ส่วน Lab นี้ใช้
> `KMeans` ของ scikit-learn ที่ใช้ k-means++ และรัน 10 รอบเลือกผลที่ดีที่สุด
>
> **ค่าที่ต้องตรงกัน:** silhouette ของแต่ละ k · ค่า k ที่ดีที่สุด · ขนาดและโปรไฟล์ของแต่ละกลุ่ม
> **ค่าที่อาจต่างในทศนิยมท้าย ๆ:** inertia — เพราะขึ้นกับค่าเริ่มต้น
>
> ความต่างนี้เองเป็นบทเรียน: **K-Means ไวต่อค่าเริ่มต้น** จึงต้องตั้ง `n_init` และ
> `random_state` เสมอ มิฉะนั้นผลจะไม่ซ้ำเดิมแม้รันบนข้อมูลชุดเดียวกัน